##### debug

In [1]:
import os 
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

## Spark Session

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.master('local[*]').getOrCreate()

## Reading data

In [3]:
df_vehicles = spark.read.option('multiLine','true').json('../data/sample_vehicles.json')

df_traffic = spark.read.option('multiLine','true').json('../data/sample_traffic.json')

df_stops= spark.read.option('multiLine','true').json('../data/sample_stops.json')

In [4]:
df_vehicles.show()

+--------------------+--------------------+
|               buses|               trams|
+--------------------+--------------------+
|[{1, 52.234654, 1...|[{015, 52.25587, ...|
+--------------------+--------------------+



#### Some variables

In [5]:
LAT_MIN, LAT_MAX = 52.05, 52.40
LON_MIN, LON_MAX = 20.70, 21.45

In [6]:
OUTPUT_DIR = '../data/cleaned_data'

## Transformations

### Busses/Trams

In [7]:
buses_df = df_vehicles.select(F.explode('buses').alias('buses'),)
buses_df = buses_df.select('buses.*')
buses_df = buses_df.withColumn('Type',F.lit('Bus'))

trams_df = df_vehicles.select(F.explode('trams').alias('trams'))
trams_df = trams_df.select('trams.*')
trams_df = trams_df.withColumn('Type',F.lit('Tram'))

vehicle_df = buses_df.unionByName(trams_df)


In [8]:
df_stops.show(1)

+--------------------+
|              values|
+--------------------+
|[{zespol, 4225}, ...|
+--------------------+
only showing top 1 row



In [9]:
vehicle_df.select('*').count()

1203

In [10]:
vehicle_df.show(5)

+-------+----------+-----+----------+-------------------+-------------+----+
|Brigade|       Lat|Lines|       Lon|               Time|VehicleNumber|Type|
+-------+----------+-----+----------+-------------------+-------------+----+
|      1| 52.234654|  161| 21.115176|2026-07-09 23:19:39|         1000| Bus|
|      2|  52.23456|  161| 21.115815|2026-07-09 22:20:14|         1001| Bus|
|      5|52.2343271|  161|21.1162238|2026-07-10 00:08:47|         1002| Bus|
|      3| 52.234606|  219| 21.115411|2026-07-09 22:37:24|         1003| Bus|
|      2| 52.234551|  311| 21.115811|2026-07-09 21:09:02|         1006| Bus|
+-------+----------+-----+----------+-------------------+-------------+----+
only showing top 5 rows



In [11]:
vehicle_df.select([
    F.sum(F.when((F.col(c).isNull()) | (F.col(c)==''),1).otherwise(0)).alias(c) for c in vehicle_df.columns
]).show()

+-------+---+-----+---+----+-------------+----+
|Brigade|Lat|Lines|Lon|Time|VehicleNumber|Type|
+-------+---+-----+---+----+-------------+----+
|      0|  0|    0|  0|   0|            0|   0|
+-------+---+-----+---+----+-------------+----+



In [12]:
vehicle_df.schema

StructType([StructField('Brigade', StringType(), True), StructField('Lat', DoubleType(), True), StructField('Lines', StringType(), True), StructField('Lon', DoubleType(), True), StructField('Time', StringType(), True), StructField('VehicleNumber', StringType(), True), StructField('Type', StringType(), False)])

In [13]:
vehicle_df = (
    vehicle_df
    .withColumn('Time',F.to_timestamp(F.col('Time'),'yyyy-MM-dd HH:mm:ss'))
)

In [14]:
vehicle_df =vehicle_df.filter(
    (F.col('Lat') >= LAT_MIN) & (F.col('Lat') <= LAT_MAX) &
    (F.col("Lon") >=LON_MIN) & (F.col('Lon') <= LON_MAX)
)

In [15]:
vehicle_df=vehicle_df.filter(
    (F.col('Time') >= F.current_timestamp()-F.expr('INTERVAL 2 HOURS'))
)

In [16]:
vehicle_df=(
    vehicle_df
    .withColumn('year',F.date_format(F.col('Time'),'yyyy'))
    .withColumn('month',F.date_format(F.col('Time'),'MM'))
    .withColumn('day',F.date_format(F.col('Time'),'dd'))
    .withColumn('hour',F.date_format(F.col('Time'),'HH'))
)

In [17]:
vehicle_df.write.mode('append').partitionBy('year','month','day','hour').parquet(f'{OUTPUT_DIR}/vehicles')

### Stops

In [18]:
stops_df = df_stops.withColumn('id',F.monotonically_increasing_id())
stops_df = stops_df.select('id',F.explode('values').alias('item'))
stops_df=stops_df.select('id',F.col('item.key').alias('key'),F.col('item.value').alias('value'))

In [19]:
stops_df.show(7)

+---+-------------+-----------+
| id|          key|      value|
+---+-------------+-----------+
|  0|       zespol|       4225|
|  0|       slupek|         03|
|  0|nazwa_zespolu|Sympatyczna|
|  0|     id_ulicy|       2336|
|  0|     szer_geo|    52.2103|
|  0|     dlug_geo|   20.91986|
|  1|       zespol|       4213|
+---+-------------+-----------+
only showing top 7 rows



In [20]:
stops_df=stops_df.groupBy('id').pivot('key').agg(F.first('value'))

In [21]:
stops_df = (stops_df
    .withColumn('lat',F.col('szer_geo').cast('Double'))
    .withColumn('lon',F.col('dlug_geo').cast('Double'))
    .filter(
        (F.col("lat").between(LAT_MIN,LAT_MAX)) &
        (F.col('lon').between(LON_MIN,LON_MAX))
    )
    .select(
        F.col('id_ulicy'),
        F.col('nazwa_zespolu'),
        F.col('slupek'),
        F.col("zespol"),
        F.col('lat'),
        F.col('lon')
    )
)

In [22]:
stops_df.show(3)

+--------+-------------+------+------+---------+---------+
|id_ulicy|nazwa_zespolu|slupek|zespol|      lat|      lon|
+--------+-------------+------+------+---------+---------+
|    2336|  Sympatyczna|    03|  4225|  52.2103| 20.91986|
|    2337|    Wałowicka|    04|  4213|52.209584|20.913975|
|    2333|    Parowcowa|    02|  4212| 52.21233| 20.91389|
+--------+-------------+------+------+---------+---------+
only showing top 3 rows



In [23]:
stops_df.write.mode('overwrite').parquet(f'{OUTPUT_DIR}/stops')

### Traffic

In [24]:
traffic_df = df_traffic.withColumn(
    'id',F.monotonically_increasing_id()
)

In [25]:
traffic_df = (
    traffic_df
    .select('*',
    F.explode('geo').alias('geo_group'))
)

In [26]:
traffic_df = (
    traffic_df
    .select('*',F.explode('geo_group').alias('geomerty'))
)

In [27]:
traffic_df=traffic_df.select(
    '*',
    F.col('geomerty.geomType').alias('geoType'),
    F.col('geomerty.latlngs').alias('latlngs')
)

In [28]:
traffic_df = (
    traffic_df
    .select('*',F.posexplode('latlngs').alias('point_order','point'))
    .select('*',F.col('point.lat').cast('Double').alias('lat'),
                F.col('point.lng').cast('Double').alias('lon')) 
).drop('geo_group','geomerty','latlngs','point','geo')


In [29]:
traffic_df =traffic_df.filter(
    (F.col('lat').between(LAT_MIN,LAT_MAX)) &
    (F.col('lon').between(LON_MIN,LON_MAX))
)

In [ ]:
traffic_df = (
    traffic_df
    .withColumn('content',F.explode('content'))
    .withColumn('detour_type',F.explode('detour_type'))
    .withColumn('streets',F.explode('streets'))
)

In [33]:
traffic_df.show(2)

+--------------------+--------------------+-------------------+---+--------------------+-------------------+--------------------+-------+-----------+-----------------+-----------------+
|             content|         detour_type|           end_date| id|                name|         start_date|             streets|geoType|point_order|              lat|              lon|
+--------------------+--------------------+-------------------+---+--------------------+-------------------+--------------------+-------+-----------+-----------------+-----------------+
|W sobotę i niedzi...|[Utrudnienia w ru...|2026-07-12 18:00:00|  0|Weekendowe zgroma...|2026-07-11 17:00:00|[Aleje Ujazdowski...|      1|          0|52.22773041160903|21.02315962432532|
|Sobotni przemarsz...|[Utrudnienia w ru...|2026-07-12 18:00:00|  0|Weekendowe zgroma...|2026-07-11 17:00:00|[Aleje Ujazdowski...|      1|          0|52.22773041160903|21.02315962432532|
+--------------------+--------------------+-------------------+---+---

In [31]:
traffic_df.write.mode('overwrite').parquet(f'{OUTPUT_DIR}/traffic')